In [2]:
# ========================================
# IMPORT LIBRARIES
# ========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, expr, size, length, trim, lower,
    year, month, dayofmonth, coalesce, when, count
)
from pyspark.sql.types import IntegerType, LongType
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ========================================
# INITIALIZE SPARK SESSION
# ========================================
print("Initializing Spark Session...")
spark = SparkSession.builder \
    .appName("YouTubeAnalytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark initialized successfully!\n")

Initializing Spark Session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/30 21:16:28 WARN Utils: Your hostname, hung-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.248.135 instead (on interface ens33)
25/10/30 21:16:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/30 21:16:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/30 21:16:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark initialized successfully!



In [4]:
from pyspark.sql.functions import col, explode

# ========================================
# 1. TẢI DỮ LIỆU THÔ (CSV) TỪ HDFS
# ========================================
print("Đang tải dữ liệu thô (raw_data.csv) từ HDFS...")

hdfs_csv_path = "hdfs://localhost:9000/data/raw_data/raw_data.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(hdfs_csv_path)

print(f"Tải xong CSV: {df.count()} dòng, {len(df.columns)} cột")

# ========================================
# 2. TẢI VÀ XỬ LÝ CATEGORY (JSON) TỪ HDFS
# ========================================
print("\nĐang tải và xử lý file Category (JSON) từ HDFS...")

hdfs_json_path = "hdfs://localhost:9000/data/raw_data/category_id.json"

# Đọc file JSON (lưu ý phải có 'multiLine=True')
df_category_lookup = spark.read.json(hdfs_json_path, multiLine=True)

# Dùng 'explode' để làm phẳng mảng 'items'
df_categories = df_category_lookup.select(explode(col("items")).alias("item")) \
                                  .select(
                                      # "id" trong file JSON chính là "categoryId"
                                      col("item.id").alias("category_id_lookup"), 
                                      col("item.snippet.title").alias("category_name")
                                  )

print("Bảng tra cứu Category đã sẵn sàng:")
df_categories.show(5)

# ========================================
# 3. KẾT HỢP (JOIN) HAI DATAFRAME
# ========================================
print("\nĐang join tên Category vào DataFrame chính...")

# Chú ý: Cột ID trong 'df' tên là 'categoryId' (theo file CSV của bạn)
# Cột ID trong 'df_categories' ta vừa đặt là 'category_id_lookup'
df_final = df.join(
    df_categories,
    df.categoryId == df_categories.category_id_lookup, # Điều kiện join
    "left" # Kiểu join (giữ lại tất cả video)
)

# Gán 'df' bằng DataFrame đã join và xóa cột ID thừa
df = df_final.drop("category_id_lookup")

print("✅ Join hoàn tất! DataFrame đã có cột 'category_name'.")
df.select("title", "categoryId", "category_name").show(5)

Đang tải dữ liệu thô (raw_data.csv) từ HDFS...


Tải xong CSV: 268787 dòng, 16 cột

Đang tải và xử lý file Category (JSON) từ HDFS...
Bảng tra cứu Category đã sẵn sàng:


+------------------+----------------+
|category_id_lookup|   category_name|
+------------------+----------------+
|                 1|Film & Animation|
|                 2|Autos & Vehicles|
|                10|           Music|
|                15|  Pets & Animals|
|                17|          Sports|
+------------------+----------------+
only showing top 5 rows

Đang join tên Category vào DataFrame chính...
✅ Join hoàn tất! DataFrame đã có cột 'category_name'.
+--------------------+----------+--------------+
|               title|categoryId| category_name|
+--------------------+----------+--------------+
|I ASKED HER TO BE...|        22|People & Blogs|
|Apex Legends | St...|        20|        Gaming|
|I left youtube fo...|        24| Entertainment|
|XXL 2020 Freshman...|        10|         Music|
|Ultimate DIY Home...|        26| Howto & Style|
+--------------------+----------+--------------+
only showing top 5 rows


In [5]:
# ========================================
# EXPLORE DATA STRUCTURE
# ========================================
print("=" * 60)
print("DATA STRUCTURE")
print("=" * 60)
df.printSchema()
print("\nSample data:")
df.show(5, truncate=50)


DATA STRUCTURE
root
 |-- video_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- publishedAt: timestamp (nullable = true)
 |-- channelId: string (nullable = true)
 |-- channelTitle: string (nullable = true)
 |-- categoryId: integer (nullable = true)
 |-- trending_date: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- view_count: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: boolean (nullable = true)
 |-- ratings_disabled: boolean (nullable = true)
 |-- description: string (nullable = true)
 |-- category_name: string (nullable = true)


Sample data:


+-----------+--------------------------------------------------+-------------------+------------------------+-------------+----------+-------------------+--------------------------------------------------+----------+------+--------+-------------+----------------------------------------------+-----------------+----------------+--------------------------------------------------+--------------+
|   video_id|                                             title|        publishedAt|               channelId| channelTitle|categoryId|      trending_date|                                              tags|view_count| likes|dislikes|comment_count|                                thumbnail_link|comments_disabled|ratings_disabled|                                       description| category_name|
+-----------+--------------------------------------------------+-------------------+------------------------+-------------+----------+-------------------+--------------------------------------------------+-----

In [6]:
# ========================================
# CHECK MISSING VALUES
# ========================================
print("\n" + "=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.show(vertical=True)


MISSING VALUES ANALYSIS


-RECORD 0-----------------
 video_id          | 0    
 title             | 0    
 publishedAt       | 0    
 channelId         | 0    
 channelTitle      | 0    
 categoryId        | 0    
 trending_date     | 0    
 tags              | 0    
 view_count        | 0    
 likes             | 0    
 dislikes          | 0    
 comment_count     | 0    
 thumbnail_link    | 0    
 comments_disabled | 0    
 ratings_disabled  | 0    
 description       | 4549 
 category_name     | 0    



In [33]:
# ========================================
# REMOVE EMPTY ROWS
# ========================================
print("\nRemoving completely empty rows...")
before = df.count()
df = df.dropna(how='all')
removed = before - df.count()
print(f"Removed: {removed} empty rows")



Removing completely empty rows...


Removed: 0 empty rows


In [34]:
# # ========================================
# # FILTER VALID TRENDING_DATE
# # ========================================
# print("\nFiltering valid trending_date format...")
# before = df.count()
# df = df.filter(col("trending_date").rlike(r"^\d{2}\.\d{2}\.\d{2}$"))
# removed = before - df.count()
# print(f"Removed: {removed} rows with invalid date format")

In [7]:
# ========================================
# REMOVE CORRUPTED VIDEO_ID
# ========================================
print("\nRemoving corrupted video_id...")
before = df.count()
df = df.filter(
    (col("video_id").isNotNull()) & 
    (col("video_id") != "#NAME?") &
    (length(col("video_id")) > 5)
)
removed = before - df.count()
print(f"Removed: {removed} invalid video_id records")


Removing corrupted video_id...


Removed: 0 invalid video_id records


In [8]:
# ========================================
# FILL MISSING DESCRIPTIONS
# ========================================
print("\nFilling missing descriptions...")
df = df.withColumn(
    "description",
    coalesce(col("description"), lit("No description available"))
)
print("Missing descriptions filled with default text")


Filling missing descriptions...


Missing descriptions filled with default text


In [9]:
# ========================================
# CONVERT DATETIME FIELDS
# ========================================
from pyspark.sql.types import TimestampType

print("\nConverting datetime fields...")
df = df.withColumn(
    "trending_date",
    col("trending_date").cast(TimestampType()) # <--- SỬA LẠI THÀNH DÒNG NÀY
)

df = df.withColumn(
    "publish_time", 
    expr("to_timestamp(publishedAt, \"yyyy-MM-dd'T'HH:mm:ss.SSS'Z'\")") # Sửa ở đây
)
print("Datetime conversion completed")


Converting datetime fields...


Datetime conversion completed


In [10]:
# ========================================
# EXTRACT DATE COMPONENTS
# ========================================
print("\nExtracting date components...")
df = df.withColumn("trending_year", year(col("trending_date"))) \
       .withColumn("trending_month", month(col("trending_date"))) \
       .withColumn("trending_day", dayofmonth(col("trending_date"))) \
       .withColumn("publish_year", year(col("publish_time"))) \
       .withColumn("publish_month", month(col("publish_time")))
print("Date components extracted: year, month, day")



Extracting date components...


Date components extracted: year, month, day


In [11]:
# ========================================
# CONVERT NUMERIC COLUMNS
# ========================================
print("\nConverting and cleaning numeric columns...")
numeric_cols = ['view_count', 'likes', 'dislikes', 'comment_count']

for col_name in numeric_cols:
    df = df.withColumn(col_name, col(col_name).cast(LongType()))
    df = df.withColumn(
        col_name,
        when(col(col_name).isNull() | (col(col_name) < 0), 0)
        .otherwise(col(col_name))
    )
print(f"Converted columns: {', '.join(numeric_cols)}")



Converting and cleaning numeric columns...
Converted columns: view_count, likes, dislikes, comment_count


In [12]:
# ========================================
# PROCESS TAGS FIELD
# ========================================
print("\nProcessing tags field...")
df = df.withColumn(
    "tags",
    when(col("tags") == "[none]", lit("")).otherwise(col("tags"))
)

df = df.withColumn(
    "tags",
    expr("SPLIT(REGEXP_REPLACE(tags, '\"', ''), '\\\\|')")
)

df = df.withColumn("tag_count", size(col("tags")))
print("Tags converted to array and counted")


Processing tags field...


Tags converted to array and counted


In [13]:
# ========================================
# CREATE ENGAGEMENT METRICS
# ========================================
print("\nCreating engagement metrics...")

# Engagement rate: (likes + dislikes + comments) / views * 100
df = df.withColumn(
    "engagement_rate",
    expr("ROUND((likes + dislikes + comment_count) / NULLIF(view_count, 0) * 100, 2)")
)

# Like ratio: likes / (likes + dislikes) * 100
df = df.withColumn(
    "like_ratio",
    expr("ROUND(likes / NULLIF(likes + dislikes, 0) * 100, 2)")
)

# Days from publish to trending
df = df.withColumn(
    "days_to_trend",
    expr("DATEDIFF(trending_date, publish_time)")
)
print("Created metrics: engagement_rate, like_ratio, days_to_trend")


Creating engagement metrics...
Created metrics: engagement_rate, like_ratio, days_to_trend


In [14]:
# ========================================
# REMOVE DUPLICATES
# ========================================
print("\nRemoving duplicate videos...")
before = df.count()
df = df.orderBy(col("trending_date").desc()).dropDuplicates(["video_id"])
removed = before - df.count()
print(f"Removed: {removed} duplicate records")



Removing duplicate videos...


Removed: 221645 duplicate records


In [15]:
# ========================================
# VALIDATE FINAL DATA
# ========================================
print("\n" + "=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)
print(f"Final dataset: {df.count()} rows, {len(df.columns)} columns")

critical_cols = ['video_id', 'trending_date', 'title', 'view_count']
print("\nNull check for critical columns:")
df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in critical_cols
]).show()



FINAL DATA VALIDATION


Final dataset: 47142 rows, 27 columns

Null check for critical columns:


+--------+-------------+-----+----------+
|video_id|trending_date|title|view_count|
+--------+-------------+-----+----------+
|       0|            0|    0|         0|
+--------+-------------+-----+----------+



In [16]:
# ========================================
# PREVIEW RESULTS
# ========================================
print("\n" + "=" * 60)
print("PREVIEW PROCESSED DATA")
print("=" * 60)
df.select(
    "video_id", "title", "channelTitle", 
    "view_count", "likes", "engagement_rate", "tag_count"
).show(10, truncate=40)



PREVIEW PROCESSED DATA


+-----------+----------------------------------------+-----------------------------+----------+------+---------------+---------+
|   video_id|                                   title|                 channelTitle|view_count| likes|engagement_rate|tag_count|
+-----------+----------------------------------------+-----------------------------+----------+------+---------------+---------+
|--47FjCWgrU|San Francisco 49ers vs. Arizona Cardi...|                          NFL|   1940781| 22612|           1.27|        1|
|--DKkzWVh-E|            Why Retaining Walls Collapse|        Practical Engineering|    623949| 29991|           5.02|       25|
|--SvHNpSvpk|YoungBoy Never Broke Again - Dead Tro...|   YoungBoy Never Broke Again|   5308719|175482|           3.68|       24|
|--gJDs10ShA|what happens if you get greedy and tr...|            Hydraulic Beanbag|   3089272|157048|           5.21|       22|
|--hjHKgm67g|     The Third Attempt Making This Table|             Blacktail Studio|    857931| 2

In [17]:
# ========================================
# STATISTICS SUMMARY
# ========================================
print("\n" + "=" * 60)
print("STATISTICS SUMMARY")
print("=" * 60)
df.select("view_count", "likes", "engagement_rate", "days_to_trend") \
  .summary("count", "mean", "min", "max", "stddev") \
  .show()



STATISTICS SUMMARY


+-------+------------------+------------------+-----------------+------------------+
|summary|        view_count|             likes|  engagement_rate|     days_to_trend|
+-------+------------------+------------------+-----------------+------------------+
|  count|             47142|             47142|            47124|             47142|
|   mean|2638239.2069916422|119321.84945483858| 5.13950980392156| 5.826418056085868|
|    min|                 0|                 0|              0.0|                 0|
|    max|         277791741|          16021534|            39.56|                37|
| stddev| 7121858.753609199| 365038.5728639505|3.511906035625927|1.9912122063291737|
+-------+------------------+------------------+-----------------+------------------+



In [ ]:
# # ========================================
# # SAVE PROCESSED DATA
# # ========================================
# print("\nSaving processed data...")
# output_path = './data/cleaned_data' 

# # 'df' BÂY GIỜ đã chứa 'category_name'
# df_to_save = df.withColumn(
#     "tags",
#     expr("array_join(tags, '|')")
# )

# df_to_save.coalesce(1).write \
#   .mode('overwrite') \
#   .option('header', 'true') \
#   .csv(output_path)

# print(f"Data saved to: {output_path}")
# print("Processing completed successfully!")


Saving processed data...


Data saved to: ./data/cleaned_data
Processing completed successfully!


In [ ]:

output_path = "hdfs://localhost:9000/data/cleaned_data"

print(f"Đang lưu dữ liệu đã xử lý vào HDFS tại: {output_path} (định dạng Parquet)...")

df_to_save = df.withColumn(
    "tags",
    expr("array_join(tags, '|')")
)

df_to_save.write \
    .mode('overwrite') \
    .parquet(output_path)

print("✅ Đã lưu thành công file Parquet!")


Đang lưu dữ liệu đã xử lý vào HDFS tại: hdfs://localhost:9000/data/cleaned_data (định dạng Parquet)...


Py4JJavaError: An error occurred while calling o465.parquet.
: org.apache.hadoop.security.AccessControlException: Permission denied: user=hung, access=WRITE, inode="/data":root:supergroup:drwxr-xr-x
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.check(FSPermissionChecker.java:399)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:255)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:193)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkPermission(FSDirectory.java:1879)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkPermission(FSDirectory.java:1863)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkAncestorAccess(FSDirectory.java:1822)
	at org.apache.hadoop.hdfs.server.namenode.FSDirMkdirOp.mkdirs(FSDirMkdirOp.java:59)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.mkdirs(FSNamesystem.java:3233)
	at org.apache.hadoop.hdfs

In [20]:

# # ========================================
# # STOP SPARK SESSION
# # ========================================
# print("\nStopping Spark session...")
# spark.stop()
# print("Done!")